# Lumbar Spine MRI Segmentation
## Modified U-Net with Combined Loss (Focal + Dice)

Based on: *Pioneering Precision in Lumbar Spine MRI Segmentation with Advanced Deep Learning and Data Enhancement* (Ahmed et al., 2025)

---

## 0. Environment Setup

In [1]:
# Google Drive mount
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Install SimpleITK for reading MHA files
!pip install SimpleITK -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 6.6 MB/s eta 0:00:00:00:0100:01m


In [3]:
import os
import numpy as np
import SimpleITK as sitk
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

# Paths
BASE_DIR = Path('/content/drive/MyDrive/SPIDER/DataSet')
IMAGE_DIR = BASE_DIR / 'images'
MASK_DIR = BASE_DIR / 'masks'

print(f'Image dir exists: {IMAGE_DIR.exists()}')
print(f'Mask dir exists: {MASK_DIR.exists()}')

Image dir exists: True
Mask dir exists: True


## 1. Data Exploration

SPIDERデータセットの構造を確認する。

### 1.1 File Listing

In [4]:
# List all files
image_files = sorted(os.listdir(IMAGE_DIR))
mask_files = sorted(os.listdir(MASK_DIR))

print(f'Number of image files: {len(image_files)}')
print(f'Number of mask files: {len(mask_files)}')
print(f'\nFirst 10 image files:')
for f in image_files[:10]:
    print(f'  {f}')
print(f'\nFirst 10 mask files:')
for f in mask_files[:10]:
    print(f'  {f}')

Number of image files: 447
Number of mask files: 447

First 10 image files:
  100_t1.mha
  100_t2.mha
  101_t1.mha
  101_t2.mha
  104_t1.mha
  104_t2.mha
  105_t1.mha
  105_t2.mha
  106_t1.mha
  106_t2.mha

First 10 mask files:
  100_t1.mha
  100_t2.mha
  101_t1.mha
  101_t2.mha
  104_t1.mha
  104_t2.mha
  105_t1.mha
  105_t2.mha
  106_t1.mha
  106_t2.mha


In [5]:
# Check file extensions
image_extensions = Counter(Path(f).suffix for f in image_files)
mask_extensions = Counter(Path(f).suffix for f in mask_files)
print(f'Image extensions: {dict(image_extensions)}')
print(f'Mask extensions: {dict(mask_extensions)}')

Image extensions: {'.mha': 447}
Mask extensions: {'.mha': 447}


In [6]:
# Check file naming pattern (T1, T2, T2_SPACE)
def classify_sequence(filename):
    name = filename.lower()
    if 't2_space' in name or 't2_sag_space' in name or 'space' in name:
        return 'T2_SPACE'
    elif 't2' in name:
        return 'T2'
    elif 't1' in name:
        return 'T1'
    else:
        return 'Unknown'

sequence_counts = Counter(classify_sequence(f) for f in image_files)
print(f'Sequence distribution: {dict(sequence_counts)}')
print(f'\nSample filenames by sequence:')
for seq in ['T1', 'T2', 'T2_SPACE', 'Unknown']:
    samples = [f for f in image_files if classify_sequence(f) == seq][:3]
    if samples:
        print(f'  {seq}: {samples}')

Sequence distribution: {'T1': 196, 'T2': 210, 'T2_SPACE': 41}

Sample filenames by sequence:
  T1: ['100_t1.mha', '101_t1.mha', '104_t1.mha']
  T2: ['100_t2.mha', '101_t2.mha', '104_t2.mha']
  T2_SPACE: ['107_t2_SPACE.mha', '110_t2_SPACE.mha', '118_t2_SPACE.mha']


### 1.2 MHA File Structure

In [7]:
# Load first image and mask to inspect
# Filter to only mha files
mha_images = [f for f in image_files if f.endswith('.mha')]
mha_masks = [f for f in mask_files if f.endswith('.mha')]

if not mha_images:
    print('No .mha files found. Listing all files for inspection:')
    for f in image_files[:20]:
        print(f'  {f}')
else:
    sample_img_path = IMAGE_DIR / mha_images[0]
    sample_mask_path = MASK_DIR / mha_masks[0]

    img = sitk.ReadImage(str(sample_img_path))
    mask = sitk.ReadImage(str(sample_mask_path))

    print(f'=== Sample: {mha_images[0]} ===')
    print(f'Image size: {img.GetSize()}')
    print(f'Image spacing: {img.GetSpacing()}')
    print(f'Image origin: {img.GetOrigin()}')
    print(f'Image direction: {img.GetDirection()}')
    print(f'Image pixel type: {img.GetPixelIDTypeAsString()}')
    print(f'\nMask size: {mask.GetSize()}')
    print(f'Mask spacing: {mask.GetSpacing()}')
    print(f'Mask pixel type: {mask.GetPixelIDTypeAsString()}')

    # Convert to numpy
    img_arr = sitk.GetArrayFromImage(img)
    mask_arr = sitk.GetArrayFromImage(mask)
    print(f'\nImage array shape: {img_arr.shape}  (slices, height, width)')
    print(f'Image dtype: {img_arr.dtype}')
    print(f'Image value range: [{img_arr.min()}, {img_arr.max()}]')
    print(f'\nMask array shape: {mask_arr.shape}')
    print(f'Mask dtype: {mask_arr.dtype}')
    print(f'Mask unique values: {np.unique(mask_arr)}')
    print(f'Mask unique count: {len(np.unique(mask_arr))}')

=== Sample: 100_t1.mha ===
Image size: (21, 492, 797)
Image spacing: (4.389695960744017, 0.6305175528662943, 0.38792314385787563)
Image origin: (-44.77723154001997, -122.57786786661, -106.77207503644567)
Image direction: (1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)
Image pixel type: 16-bit signed integer

Mask size: (21, 492, 797)
Mask spacing: (4.389695960744017, 0.6305175528662943, 0.38792314385787563)
Mask pixel type: 16-bit signed integer

Image array shape: (797, 492, 21)  (slices, height, width)
Image dtype: int16
Image value range: [-1000, 3096]

Mask array shape: (797, 492, 21)
Mask dtype: int16
Mask unique values: [  0   1   2   3   4   5   6   7   8 100 201 202 203 204 205 206 207 208]
Mask unique count: 18


### 1.3 Label Value Investigation

全マスクのラベル値を確認し、16クラス→4クラスのマッピングを決定する。

In [8]:
# Check label values across multiple masks
all_label_values = set()
label_info = []

for i, fname in enumerate(mha_masks[:20]):  # Check first 20 masks
    mask = sitk.ReadImage(str(MASK_DIR / fname))
    mask_arr = sitk.GetArrayFromImage(mask)
    unique_vals = np.unique(mask_arr)
    all_label_values.update(unique_vals.tolist())
    label_info.append({
        'file': fname,
        'shape': mask_arr.shape,
        'unique_values': unique_vals.tolist(),
        'n_labels': len(unique_vals)
    })
    print(f'{fname}: shape={mask_arr.shape}, labels={unique_vals}, count={len(unique_vals)}')

print(f'\n=== All unique label values across 20 masks ===')
print(sorted(all_label_values))
print(f'Total unique labels: {len(all_label_values)}')

100_t1.mha: shape=(797, 492, 21), labels=[  0   1   2   3   4   5   6   7   8 100 201 202 203 204 205 206 207 208], count=18
100_t2.mha: shape=(797, 492, 21), labels=[  0   1   2   3   4   5   6   7   8 100 201 202 203 204 205 206 207 208], count=18
101_t1.mha: shape=(298, 320, 17), labels=[  0   1   2   3   4   5   6 100 201 202 203 204 205 206], count=14
101_t2.mha: shape=(352, 384, 17), labels=[  0   1   2   3   4   5   6 100 201 202 203 204 205 206], count=14
104_t1.mha: shape=(320, 320, 15), labels=[  0   1   2   3   4   5   6   7 100 201 202 203 204 205 206 207], count=16
104_t2.mha: shape=(384, 384, 15), labels=[  0   1   2   3   4   5   6   7 100 201 202 203 204 205 206 207], count=16
105_t1.mha: shape=(427, 448, 25), labels=[  0   1   2   3   4   5   6   7 100 201 202 203 204 205 206 207], count=16
105_t2.mha: shape=(427, 448, 25), labels=[  0   1   2   3   4   5   6   7 100 201 202 203 204 205 206 207], count=16
106_t1.mha: shape=(384, 384, 15), labels=[  0   1   2   3   4   

In [ ]:
# Detailed label frequency for ONE mask
if mha_masks:
    mask = sitk.ReadImage(str(MASK_DIR / mha_masks[0]))
    mask_arr = sitk.GetArrayFromImage(mask)
    unique, counts = np.unique(mask_arr, return_counts=True)
    total = mask_arr.size

    print(f'=== Label distribution: {mha_masks[0]} ===')
    print(f'{"Label":>8} {"Count":>12} {"Percentage":>12}')
    print('-' * 35)
    for val, cnt in zip(unique, counts):
        print(f'{val:>8} {cnt:>12} {cnt/total*100:>11.2f}%')

### 1.4 Visualize Sample Slices

In [ ]:
# Visualize middle slices of first few samples
if mha_images:
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))

    for row in range(3):
        if row >= len(mha_images):
            break
        img = sitk.ReadImage(str(IMAGE_DIR / mha_images[row]))
        mask = sitk.ReadImage(str(MASK_DIR / mha_masks[row]))
        img_arr = sitk.GetArrayFromImage(img)
        mask_arr = sitk.GetArrayFromImage(mask)

        mid_slice = img_arr.shape[0] // 2

        # Original image
        axes[row, 0].imshow(img_arr[mid_slice], cmap='gray')
        axes[row, 0].set_title(f'{mha_images[row]}\nSlice {mid_slice}')
        axes[row, 0].axis('off')

        # Mask (all labels)
        axes[row, 1].imshow(mask_arr[mid_slice], cmap='nipy_spectral')
        axes[row, 1].set_title(f'Mask (all labels)\nUnique: {np.unique(mask_arr[mid_slice])}')
        axes[row, 1].axis('off')

        # Mask overlaid on image
        axes[row, 2].imshow(img_arr[mid_slice], cmap='gray')
        axes[row, 2].imshow(mask_arr[mid_slice], cmap='nipy_spectral', alpha=0.4)
        axes[row, 2].set_title('Overlay')
        axes[row, 2].axis('off')

        # Histogram of label values in this slice
        vals, cnts = np.unique(mask_arr[mid_slice], return_counts=True)
        axes[row, 3].bar([str(v) for v in vals], cnts)
        axes[row, 3].set_title('Label distribution')
        axes[row, 3].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()

In [ ]:
# Show multiple slices of one volume to understand 3D structure
if mha_images:
    img = sitk.ReadImage(str(IMAGE_DIR / mha_images[0]))
    mask = sitk.ReadImage(str(MASK_DIR / mha_masks[0]))
    img_arr = sitk.GetArrayFromImage(img)
    mask_arr = sitk.GetArrayFromImage(mask)

    n_slices = img_arr.shape[0]
    indices = np.linspace(0, n_slices - 1, min(8, n_slices), dtype=int)

    fig, axes = plt.subplots(2, len(indices), figsize=(20, 6))
    for i, idx in enumerate(indices):
        axes[0, i].imshow(img_arr[idx], cmap='gray')
        axes[0, i].set_title(f'Slice {idx}')
        axes[0, i].axis('off')

        axes[1, i].imshow(mask_arr[idx], cmap='nipy_spectral')
        axes[1, i].set_title(f'Labels: {np.unique(mask_arr[idx])}')
        axes[1, i].axis('off')

    plt.suptitle(f'{mha_images[0]} - All slices overview ({n_slices} total)', fontsize=14)
    plt.tight_layout()
    plt.show()

### 1.5 Summary Statistics

In [ ]:
# Collect shape/spacing info - header only (fast)
print('Collecting metadata (header only, no pixel data)...\n')

metadata = []
for fname in mha_images:
    reader = sitk.ImageFileReader()
    reader.SetFileName(str(IMAGE_DIR / fname))
    reader.ReadImageInformation()
    metadata.append({
        'file': fname,
        'sequence': classify_sequence(fname),
        'size': reader.GetSize(),
        'spacing': tuple(round(s, 3) for s in reader.GetSpacing()),
    })

# Summary by sequence type
for seq in ['T1', 'T2', 'T2_SPACE', 'Unknown']:
    seq_data = [m for m in metadata if m['sequence'] == seq]
    if not seq_data:
        continue
    sizes = [m['size'] for m in seq_data]
    spacings = [m['spacing'] for m in seq_data]
    print(f'=== {seq} ({len(seq_data)} files) ===')
    print(f'  Size range: {min(sizes)} - {max(sizes)}')
    print(f'  Spacing range: {min(spacings)} - {max(spacings)}')
    print()

### 1.6 Check for Overview CSV

SPIDERデータセットにはtrain/val分割やメタデータを含むCSVがあるはず。

In [ ]:
# Look for CSV or other metadata files
spider_root = Path('/content/drive/MyDrive/SPIDER')
print('Files in SPIDER root:')
for f in sorted(spider_root.rglob('*')):
    if f.is_file() and not f.name.startswith('.'):
        rel = f.relative_to(spider_root)
        print(f'  {rel}  ({f.stat().st_size / 1024:.1f} KB)')

In [ ]:
# If overview CSV exists, load and display
import glob
csv_files = list(spider_root.rglob('*.csv')) + list(spider_root.rglob('*.json'))
print(f'Found metadata files: {[str(f.relative_to(spider_root)) for f in csv_files]}')

if csv_files:
    import pandas as pd
    for csv_path in csv_files:
        if csv_path.suffix == '.csv':
            df = pd.read_csv(csv_path)
            print(f'\n=== {csv_path.name} ===')
            print(f'Shape: {df.shape}')
            print(f'Columns: {list(df.columns)}')
            print(df.head(10))

---
## 2. Data Preprocessing

3D MHA → 2Dスライス抽出 → 4クラスマッピング → フィルタリング

### 2.1 Configuration

In [ ]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import cv2
import warnings
warnings.filterwarnings('ignore')

# === Paths ===
BASE_DIR = Path('/content/drive/MyDrive/SPIDER/DataSet')
IMAGE_DIR = BASE_DIR / 'images'
MASK_DIR = BASE_DIR / 'masks'
CSV_PATH = BASE_DIR / 'SPIDER Lumbar Spine Segmentation Overview.csv'

OUTPUT_DIR = Path('/content/drive/MyDrive/SPIDER/processed')
OUTPUT_IMG_DIR = OUTPUT_DIR / 'images'
OUTPUT_MASK_DIR = OUTPUT_DIR / 'masks'
OUTPUT_IMG_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_MASK_DIR.mkdir(parents=True, exist_ok=True)

# === Hyperparameters (from paper) ===
TARGET_H, TARGET_W = 512, 640        # Paper: 512x640 pixels
NUM_CLASSES = 4                       # Background, Vertebrae, Spinal Canal, IVDs
CLASS_IMBALANCE_THRESHOLD = 0.55      # Paper: 55%
MIN_CLASSES_REQUIRED = 4              # Paper: exclude slices with < 4 classes

# === Label Mapping (confirmed from data exploration) ===
# 0       -> 0 (Background)
# 1-99    -> 1 (Vertebrae)
# 100     -> 2 (Spinal Canal)
# 200+    -> 3 (IVDs)
def map_labels(mask):
    """Map SPIDER labels (18 unique) to 4 classes."""
    new_mask = np.zeros_like(mask, dtype=np.uint8)
    new_mask[(mask >= 1) & (mask <= 99)] = 1    # Vertebrae
    new_mask[mask == 100] = 2                    # Spinal Canal
    new_mask[mask >= 200] = 3                    # IVDs
    return new_mask

print('Configuration loaded.')
print(f'Output directory: {OUTPUT_DIR}')

### 2.2 Extract 2D Slices & Apply Label Mapping

3D MHA から矢状断の2Dスライスを抽出し、18ラベル→4クラスに変換して保存する。

In [ ]:
def extract_and_save_slices(image_dir, mask_dir, output_img_dir, output_mask_dir,
                            target_h, target_w):
    """Extract 2D sagittal slices from 3D MHA, map labels, resize, and save as .npz."""

    mha_files = sorted([f for f in os.listdir(image_dir) if f.endswith('.mha')])
    stats = {'total_slices': 0, 'files_processed': 0, 'errors': []}

    for fname in tqdm(mha_files, desc='Extracting slices'):
        mask_path = mask_dir / fname
        if not mask_path.exists():
            stats['errors'].append(f'No mask for {fname}')
            continue

        try:
            img_sitk = sitk.ReadImage(str(image_dir / fname))
            mask_sitk = sitk.ReadImage(str(mask_path))

            img_arr = sitk.GetArrayFromImage(img_sitk).astype(np.float32)
            mask_arr = sitk.GetArrayFromImage(mask_sitk).astype(np.int16)

            # SimpleITK returns (z, y, x) — for sagittal MRI, axis 0 is the slice axis
            n_slices = img_arr.shape[0]
            base_name = fname.replace('.mha', '')

            for s in range(n_slices):
                img_slice = img_arr[s]    # (H, W)
                mask_slice = mask_arr[s]  # (H, W)

                # Skip empty slices
                if img_slice.max() == img_slice.min():
                    continue

                # Map 18 labels -> 4 classes
                mask_4class = map_labels(mask_slice)

                # Resize to target
                img_resized = cv2.resize(img_slice, (target_w, target_h),
                                         interpolation=cv2.INTER_LINEAR)
                mask_resized = cv2.resize(mask_4class, (target_w, target_h),
                                          interpolation=cv2.INTER_NEAREST)

                # Save as compressed npz
                slice_name = f'{base_name}_s{s:03d}'
                np.savez_compressed(
                    output_img_dir / f'{slice_name}.npz', image=img_resized)
                np.savez_compressed(
                    output_mask_dir / f'{slice_name}.npz', mask=mask_resized)
                stats['total_slices'] += 1

            stats['files_processed'] += 1

        except Exception as e:
            stats['errors'].append(f'{fname}: {str(e)}')

    return stats

# Run extraction
stats = extract_and_save_slices(IMAGE_DIR, MASK_DIR, OUTPUT_IMG_DIR, OUTPUT_MASK_DIR,
                                 TARGET_H, TARGET_W)
print(f"\nFiles processed: {stats['files_processed']}")
print(f"Total slices extracted: {stats['total_slices']}")
if stats['errors']:
    print(f"Errors ({len(stats['errors'])}):")
    for e in stats['errors'][:10]:
        print(f'  {e}')

### 2.3 Data Filtering

論文に従い、以下の基準でスライスをフィルタリング:
1. 4クラス未満のスライスを除外
2. クラス不均衡比率 > 55% のスライスを除外

In [ ]:
def compute_class_imbalance_ratio(mask):
    """Compute class imbalance ratio: max_class_weight / min_class_weight.
    Only considers non-background classes that are present."""
    unique, counts = np.unique(mask, return_counts=True)
    total = mask.size

    # Class weights for non-background classes present in the mask
    non_bg = [(u, c / total) for u, c in zip(unique, counts) if u != 0]
    if len(non_bg) < 2:
        return 1.0  # Cannot compute ratio with <2 classes

    weights = [w for _, w in non_bg]
    return max(weights) / min(weights) if min(weights) > 0 else float('inf')


def filter_slices(img_dir, mask_dir, min_classes, imbalance_threshold):
    """Filter slices based on class count and imbalance ratio."""
    mask_files = sorted([f for f in os.listdir(mask_dir) if f.endswith('.npz')])

    kept = []
    removed_class_count = 0
    removed_imbalance = 0

    for fname in tqdm(mask_files, desc='Filtering slices'):
        mask = np.load(mask_dir / fname)['mask']
        unique_classes = np.unique(mask)

        # Filter 1: Must have all 4 classes
        if len(unique_classes) < min_classes:
            removed_class_count += 1
            continue

        # Filter 2: Class imbalance ratio
        ratio = compute_class_imbalance_ratio(mask)
        if ratio > (1.0 / (1.0 - imbalance_threshold)):
            # Paper uses "class weight ratio > 55%" — the highest class weight
            # among non-background exceeds 55% of the total non-background pixels
            non_bg_mask = mask[mask > 0]
            if len(non_bg_mask) > 0:
                _, counts = np.unique(non_bg_mask, return_counts=True)
                max_weight = counts.max() / counts.sum()
                if max_weight > imbalance_threshold:
                    removed_imbalance += 1
                    continue

        kept.append(fname)

    return kept, removed_class_count, removed_imbalance


# Run filtering
kept_files, rm_class, rm_imbalance = filter_slices(
    OUTPUT_IMG_DIR, OUTPUT_MASK_DIR, MIN_CLASSES_REQUIRED, CLASS_IMBALANCE_THRESHOLD)

print(f'\n=== Filtering Results ===')
print(f'Total slices before filtering: {len(os.listdir(OUTPUT_MASK_DIR))}')
print(f'Removed (< {MIN_CLASSES_REQUIRED} classes): {rm_class}')
print(f'Removed (imbalance > {CLASS_IMBALANCE_THRESHOLD*100:.0f}%): {rm_imbalance}')
print(f'Kept: {len(kept_files)}')

# Save filtered file list
FILTERED_LIST_PATH = OUTPUT_DIR / 'filtered_files.txt'
with open(FILTERED_LIST_PATH, 'w') as f:
    for fname in kept_files:
        f.write(fname + '\n')
print(f'\nFiltered file list saved to: {FILTERED_LIST_PATH}')

### 2.4 Train/Val Split & Verify Preprocessed Data

In [ ]:
# Split using SPIDER's official train/val assignment
overview_df = pd.read_csv(CSV_PATH)
train_ids = set(overview_df[overview_df['subset'] == 'training']['new_file_name'].values)
val_ids = set(overview_df[overview_df['subset'] == 'validation']['new_file_name'].values)

def get_series_id(slice_filename):
    """Extract series ID from slice filename: '100_t1_s005.npz' -> '100_t1'"""
    return '_'.join(slice_filename.replace('.npz', '').rsplit('_s', 1)[0].split('_'))

train_slices = [f for f in kept_files if get_series_id(f) in train_ids]
val_slices = [f for f in kept_files if get_series_id(f) in val_ids]

print(f'=== Train/Val Split ===')
print(f'Training slices: {len(train_slices)}')
print(f'Validation slices: {len(val_slices)}')
print(f'Unmatched: {len(kept_files) - len(train_slices) - len(val_slices)}')

# Verify: show a few preprocessed samples
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
CLASS_COLORS = {0: [0, 0, 0], 1: [255, 0, 0], 2: [0, 255, 0], 3: [0, 0, 255]}
CLASS_NAMES = {0: 'Background', 1: 'Vertebrae', 2: 'Spinal Canal', 3: 'IVDs'}

for i in range(3):
    idx = len(train_slices) // 4 * (i + 1)  # Sample evenly
    fname = train_slices[idx]

    img = np.load(OUTPUT_IMG_DIR / fname)['image']
    mask = np.load(OUTPUT_MASK_DIR / fname)['mask']

    # Normalize image for display
    img_disp = (img - img.min()) / (img.max() - img.min() + 1e-8)

    # Color mask
    color_mask = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for cls, color in CLASS_COLORS.items():
        color_mask[mask == cls] = color

    axes[i, 0].imshow(img_disp, cmap='gray')
    axes[i, 0].set_title(fname.replace('.npz', ''))
    axes[i, 0].axis('off')

    axes[i, 1].imshow(color_mask)
    axes[i, 1].set_title(f'Mask (4 classes)\nUnique: {np.unique(mask)}')
    axes[i, 1].axis('off')

    axes[i, 2].imshow(img_disp, cmap='gray')
    axes[i, 2].imshow(color_mask, alpha=0.4)
    axes[i, 2].set_title('Overlay')
    axes[i, 2].axis('off')

# Legend
legend_text = ' | '.join([f'{name}: {[c for c in color]}' for name, color
                           in zip(CLASS_NAMES.values(), CLASS_COLORS.values())])
plt.suptitle(f'Preprocessed samples — {legend_text}', fontsize=11)
plt.tight_layout()
plt.show()

---
## 3. Model Definition

Modified U-Net with Leaky ReLU, Glorot Uniform initializer, and custom Combined Loss.

### 3.1 Loss Functions

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, backend as K

# === Loss Functions (from paper) ===
# Combined Loss = alpha * Focal + (1 - alpha) * Dice
# alpha = 0.6, gamma = 4.0

def focal_loss(y_true, y_pred, gamma=4.0):
    """Focal Loss for multi-class segmentation."""
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
    focal = -y_true * tf.pow(1.0 - y_pred, gamma) * tf.math.log(y_pred)
    return tf.reduce_mean(tf.reduce_sum(focal, axis=-1))

def dice_loss(y_true, y_pred, epsilon=1e-6):
    """Dice Loss for multi-class segmentation."""
    numerator = 2.0 * tf.reduce_sum(y_true * y_pred, axis=(1, 2)) + epsilon
    denominator = tf.reduce_sum(y_true + y_pred, axis=(1, 2)) + epsilon
    dice = numerator / denominator
    return 1.0 - tf.reduce_mean(dice)

def combined_loss(alpha=0.6, gamma=4.0):
    """Combined Focal + Dice Loss."""
    def loss_fn(y_true, y_pred):
        fl = focal_loss(y_true, y_pred, gamma=gamma)
        dl = dice_loss(y_true, y_pred)
        return alpha * fl + (1.0 - alpha) * dl
    return loss_fn

print('Loss functions defined.')
print(f'Combined Loss: {0.6} * Focal(gamma=4.0) + {0.4} * Dice')

### 3.2 Modified U-Net Architecture

In [ ]:
def conv_block(x, filters, dropout_rate=0.5):
    """Conv -> BN -> LeakyReLU -> Conv -> BN -> LeakyReLU -> Dropout"""
    x = layers.Conv2D(filters, 3, padding='same',
                       kernel_initializer='glorot_uniform')(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.1)(x)
    x = layers.Conv2D(filters, 3, padding='same',
                       kernel_initializer='glorot_uniform')(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.1)(x)
    x = layers.Dropout(dropout_rate)(x)
    return x

def upsample_block(x, skip, filters):
    """Conv2DTranspose -> LeakyReLU -> Concatenate with skip connection"""
    x = layers.Conv2DTranspose(filters, 2, strides=2, padding='same',
                                kernel_initializer='glorot_uniform')(x)
    x = layers.LeakyReLU(alpha=0.1)(x)
    x = layers.Concatenate()([x, skip])
    return x

def build_modified_unet(input_shape=(512, 640, 1), num_classes=4, dropout_rate=0.5):
    """Modified U-Net as described in the paper."""
    inputs = layers.Input(shape=input_shape)

    # --- Encoder (Contractive Path) ---
    # Level 1: 64
    e1 = conv_block(inputs, 64, dropout_rate)
    p1 = layers.MaxPooling2D(2)(e1)

    # Level 2: 128
    e2 = conv_block(p1, 128, dropout_rate)
    p2 = layers.MaxPooling2D(2)(e2)

    # Level 3: 256
    e3 = conv_block(p2, 256, dropout_rate)
    p3 = layers.MaxPooling2D(2)(e3)

    # Level 4: 512
    e4 = conv_block(p3, 512, dropout_rate)
    p4 = layers.MaxPooling2D(2)(e4)

    # --- Bottleneck (extra 512-channel layer from paper) ---
    bottleneck = conv_block(p4, 512, dropout_rate)

    # --- Decoder (Expansive Path) ---
    # Level 4: 512 -> 256
    d4 = upsample_block(bottleneck, e4, 512)
    d4 = conv_block(d4, 256, dropout_rate)

    # Level 3: 256 -> 128
    d3 = upsample_block(d4, e3, 256)
    d3 = conv_block(d3, 128, dropout_rate)

    # Level 2: 128 -> 64
    d2 = upsample_block(d3, e2, 128)
    d2 = conv_block(d2, 64, dropout_rate)

    # Level 1: 64 -> 64
    d1 = upsample_block(d2, e1, 64)
    d1 = conv_block(d1, 64, dropout_rate)

    # --- Output ---
    outputs = layers.Conv2D(num_classes, 1, activation='softmax',
                             kernel_initializer='glorot_uniform')(d1)

    model = Model(inputs, outputs, name='Modified_UNet')
    return model

# Build and show summary
model = build_modified_unet(
    input_shape=(TARGET_H, TARGET_W, 1),
    num_classes=NUM_CLASSES
)
model.summary()
print(f'\nTotal parameters: {model.count_params():,}')

---
## 4. Dataset & Training

### 4.1 TensorFlow Dataset Pipeline

In [ ]:
BATCH_SIZE = 8
EPOCHS = 100

def load_sample(fname, img_dir, mask_dir, num_classes):
    """Load a single image-mask pair from npz files."""
    img = np.load(img_dir / fname)['image'].astype(np.float32)
    mask = np.load(mask_dir / fname)['mask'].astype(np.int32)

    # Normalize image to [0, 1]
    img_min, img_max = img.min(), img.max()
    if img_max > img_min:
        img = (img - img_min) / (img_max - img_min)

    # Add channel dimension: (H, W) -> (H, W, 1)
    img = img[..., np.newaxis]

    # One-hot encode mask: (H, W) -> (H, W, num_classes)
    mask_onehot = np.eye(num_classes, dtype=np.float32)[mask]

    return img, mask_onehot

def create_dataset(file_list, img_dir, mask_dir, num_classes, batch_size, shuffle=True):
    """Create a tf.data.Dataset from file list."""
    def generator():
        indices = np.arange(len(file_list))
        if shuffle:
            np.random.shuffle(indices)
        for i in indices:
            yield load_sample(file_list[i], img_dir, mask_dir, num_classes)

    ds = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(TARGET_H, TARGET_W, 1), dtype=tf.float32),
            tf.TensorSpec(shape=(TARGET_H, TARGET_W, num_classes), dtype=tf.float32),
        )
    )
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

# Create datasets
train_ds = create_dataset(train_slices, OUTPUT_IMG_DIR, OUTPUT_MASK_DIR,
                           NUM_CLASSES, BATCH_SIZE, shuffle=True)
val_ds = create_dataset(val_slices, OUTPUT_IMG_DIR, OUTPUT_MASK_DIR,
                         NUM_CLASSES, BATCH_SIZE, shuffle=False)

# Verify shapes
for img_batch, mask_batch in train_ds.take(1):
    print(f'Image batch shape: {img_batch.shape}')
    print(f'Mask batch shape:  {mask_batch.shape}')
    print(f'Image range: [{img_batch.numpy().min():.3f}, {img_batch.numpy().max():.3f}]')
    print(f'Mask sum check (should be 1.0): {mask_batch.numpy()[0, 0, 0].sum()}')

### 4.2 Metrics

In [ ]:
def mean_iou(y_true, y_pred):
    """Mean IoU metric for Keras."""
    y_pred_argmax = tf.argmax(y_pred, axis=-1)
    y_true_argmax = tf.argmax(y_true, axis=-1)

    iou_sum = 0.0
    for c in range(NUM_CLASSES):
        pred_c = tf.cast(tf.equal(y_pred_argmax, c), tf.float32)
        true_c = tf.cast(tf.equal(y_true_argmax, c), tf.float32)
        intersection = tf.reduce_sum(pred_c * true_c)
        union = tf.reduce_sum(pred_c) + tf.reduce_sum(true_c) - intersection
        iou_sum += (intersection + 1e-7) / (union + 1e-7)

    return iou_sum / NUM_CLASSES

def dice_coefficient(y_true, y_pred):
    """Dice coefficient metric for Keras."""
    y_pred_argmax = tf.argmax(y_pred, axis=-1)
    y_true_argmax = tf.argmax(y_true, axis=-1)

    dice_sum = 0.0
    for c in range(NUM_CLASSES):
        pred_c = tf.cast(tf.equal(y_pred_argmax, c), tf.float32)
        true_c = tf.cast(tf.equal(y_true_argmax, c), tf.float32)
        intersection = tf.reduce_sum(pred_c * true_c)
        dice_sum += (2.0 * intersection + 1e-7) / (tf.reduce_sum(pred_c) + tf.reduce_sum(true_c) + 1e-7)

    return dice_sum / NUM_CLASSES

print('Metrics defined: mean_iou, dice_coefficient')

### 4.3 Compile & Train

In [ ]:
# Compile model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=combined_loss(alpha=0.6, gamma=4.0),
    metrics=['accuracy', mean_iou, dice_coefficient]
)

# Callbacks
CHECKPOINT_DIR = Path('/content/drive/MyDrive/SPIDER/checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(CHECKPOINT_DIR / 'best_model.keras'),
        monitor='val_mean_iou',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_mean_iou',
        mode='max',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_mean_iou',
        mode='max',
        factor=0.5,
        patience=7,
        min_lr=1e-7,
        verbose=1
    ),
]

print('Model compiled. Ready to train.')
print(f'Checkpoints will be saved to: {CHECKPOINT_DIR}')

In [ ]:
# Train
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

# Save final model
model.save(str(CHECKPOINT_DIR / 'final_model.keras'))
print(f'Training complete. Model saved to {CHECKPOINT_DIR}')

---
## 5. Evaluation & Visualization

### 5.1 Training Curves

In [ ]:
def plot_training_curves(history):
    """Plot accuracy, dice, mean IoU, and loss curves."""
    metrics = {
        'Accuracy': ('accuracy', 'val_accuracy'),
        'Dice Coefficient': ('dice_coefficient', 'val_dice_coefficient'),
        'Mean IoU': ('mean_iou', 'val_mean_iou'),
        'Loss': ('loss', 'val_loss'),
    }

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    for ax, (title, (train_key, val_key)) in zip(axes.flat, metrics.items()):
        if train_key in history.history:
            ax.plot(history.history[train_key], label='Train')
            ax.plot(history.history[val_key], label='Validation')
            ax.set_title(title)
            ax.set_xlabel('Epoch')
            ax.set_ylabel(title)
            ax.legend()
            ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(str(CHECKPOINT_DIR / 'training_curves.png'), dpi=150)
    plt.show()
    print(f'Training curves saved to {CHECKPOINT_DIR / "training_curves.png"}')

plot_training_curves(history)

### 5.2 Class-wise Evaluation on Validation Set

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluate_classwise(model, file_list, img_dir, mask_dir, num_classes):
    """Compute class-wise Dice, IoU, Precision, Recall, F1."""
    class_names = ['Background', 'Vertebrae', 'Spinal Canal', 'IVDs']
    all_dice = {c: [] for c in range(num_classes)}
    all_iou = {c: [] for c in range(num_classes)}
    all_preds = []
    all_trues = []

    for fname in tqdm(file_list, desc='Evaluating'):
        img, mask_oh = load_sample(fname, img_dir, mask_dir, num_classes)
        pred = model.predict(img[np.newaxis, ...], verbose=0)[0]

        pred_cls = np.argmax(pred, axis=-1).flatten()
        true_cls = np.argmax(mask_oh, axis=-1).flatten()
        all_preds.append(pred_cls)
        all_trues.append(true_cls)

        # Per-class Dice and IoU
        for c in range(num_classes):
            pred_c = (pred_cls == c).astype(float)
            true_c = (true_cls == c).astype(float)
            intersection = (pred_c * true_c).sum()
            union = pred_c.sum() + true_c.sum() - intersection

            dice = (2.0 * intersection + 1e-7) / (pred_c.sum() + true_c.sum() + 1e-7)
            iou = (intersection + 1e-7) / (union + 1e-7)
            all_dice[c].append(dice)
            all_iou[c].append(iou)

    # Aggregate
    all_preds = np.concatenate(all_preds)
    all_trues = np.concatenate(all_trues)

    print(f'\n{"Class":<15} {"Dice":>8} {"IoU":>8} {"Precision":>10} {"Recall":>8} {"F1":>8}')
    print('-' * 60)
    for c in range(num_classes):
        d = np.mean(all_dice[c])
        iou = np.mean(all_iou[c])
        p = precision_score(all_trues == c, all_preds == c, zero_division=0)
        r = recall_score(all_trues == c, all_preds == c, zero_division=0)
        f1 = f1_score(all_trues == c, all_preds == c, zero_division=0)
        print(f'{class_names[c]:<15} {d:>8.4f} {iou:>8.4f} {p:>10.4f} {r:>8.4f} {f1:>8.4f}')

    mean_dice = np.mean([np.mean(all_dice[c]) for c in range(num_classes)])
    mean_iou_val = np.mean([np.mean(all_iou[c]) for c in range(num_classes)])
    print(f'\n{"Mean":<15} {mean_dice:>8.4f} {mean_iou_val:>8.4f}')

    return all_dice, all_iou

# Run evaluation
val_dice, val_iou = evaluate_classwise(
    model, val_slices[:100],  # Evaluate on first 100 val slices for speed
    OUTPUT_IMG_DIR, OUTPUT_MASK_DIR, NUM_CLASSES
)

### 5.3 Prediction Visualization

MRI画像にセグメンテーション結果をオーバーレイして表示。

In [ ]:
def visualize_predictions(model, file_list, img_dir, mask_dir, num_classes, n_samples=6):
    """Visualize predictions: Original | Ground Truth | Prediction | Overlay"""
    CLASS_COLORS = np.array([[0, 0, 0], [255, 0, 0], [0, 255, 0], [0, 0, 255]], dtype=np.uint8)
    CLASS_NAMES = ['BG', 'Vertebrae', 'Spinal Canal', 'IVDs']

    indices = np.linspace(0, len(file_list) - 1, n_samples, dtype=int)
    fig, axes = plt.subplots(n_samples, 4, figsize=(16, 4 * n_samples))

    for row, idx in enumerate(indices):
        fname = file_list[idx]
        img, mask_oh = load_sample(fname, img_dir, mask_dir, num_classes)

        pred = model.predict(img[np.newaxis, ...], verbose=0)[0]
        pred_cls = np.argmax(pred, axis=-1)
        true_cls = np.argmax(mask_oh, axis=-1)

        img_disp = img[..., 0]
        gt_color = CLASS_COLORS[true_cls]
        pred_color = CLASS_COLORS[pred_cls]

        # Original
        axes[row, 0].imshow(img_disp, cmap='gray')
        axes[row, 0].set_title('MRI' if row == 0 else '')
        axes[row, 0].axis('off')

        # Ground Truth
        axes[row, 1].imshow(gt_color)
        axes[row, 1].set_title('Ground Truth' if row == 0 else '')
        axes[row, 1].axis('off')

        # Prediction
        axes[row, 2].imshow(pred_color)
        axes[row, 2].set_title('Prediction' if row == 0 else '')
        axes[row, 2].axis('off')

        # Overlay
        axes[row, 3].imshow(img_disp, cmap='gray')
        axes[row, 3].imshow(pred_color, alpha=0.4)
        axes[row, 3].set_title('Overlay' if row == 0 else '')
        axes[row, 3].axis('off')

        # Add filename
        axes[row, 0].set_ylabel(fname.replace('.npz', ''), fontsize=8)

    # Color legend
    legend = ' | '.join([f'{n}: {c}' for n, c in
                          zip(CLASS_NAMES, ['Black', 'Red', 'Green', 'Blue'])])
    plt.suptitle(f'Segmentation Results — {legend}', fontsize=12)
    plt.tight_layout()
    plt.savefig(str(CHECKPOINT_DIR / 'predictions.png'), dpi=150)
    plt.show()

visualize_predictions(model, val_slices, OUTPUT_IMG_DIR, OUTPUT_MASK_DIR, NUM_CLASSES)